# Агрегирование точечных данных по регулярной сетке

В предыдущем разделе мы агрегировали данные по административным единицам. Но такие единицы сильно различаются по размеру и форме, а результат зависит от того, как проведены их границы (MAUP).

Альтернатива — **регулярная сетка**: разбиение территории на одинаковые ячейки. Ячейки имеют одинаковую площадь, поэтому их легко сравнивать между собой.

В этом разделе мы рассчитаем плотность кафе в Центральном районе Санкт-Петербурга:

- построим регулярную сетку квадратных ячеек;
- подсчитаем количество кафе в каждой ячейке;
- рассчитаем плотность кафе (количество на км²) и посмотрим, как размер ячейки влияет на результат.


## 0. Импорт библиотек и подготовка данных

### 0.1 Импорт библиотек


In [ ]:
import geopandas as gpd
import osmnx as ox
import matplotlib.pyplot as plt

### 0.2 Подготовка данных

Загрузим из OpenStreetMap границу Центрального района и кафе:


In [ ]:
area_name = "Центральный район, Санкт-Петербург"

admin_border = ox.geocode_to_gdf(area_name)
cafes = ox.features_from_place(area_name, {"amenity": "cafe"})

In [ ]:
# Альтернатива: загрузка из сохранённого файла (если osmnx недоступен)
# Скачать файл osm_data.gpkg: https://drive.google.com/file/d/1uhnwDvDwZHhUUum04cp7okzWcvVoQ43Y/view?usp=sharing

# admin_border = gpd.read_file("../data/osm_data.gpkg", layer="central_district")
# cafes = gpd.read_file("../data/osm_data.gpkg", layer="cafes")

In [ ]:
cafes["osm_id"] = cafes.index.get_level_values("id")
cafes = cafes[["osm_id", "cuisine", "name", "geometry"]]

В OpenStreetMap объект может быть нанесён и точкой, и полигоном (например, кафе, занимающее отдельное здание). Проверим типы геометрии:


In [ ]:
cafes.geom_type.value_counts()

Часть кафе — полигоны. Полигон, который пересекает границу ячейки, не находится целиком **внутри** (`within`) ни одной ячейки и потеряется при подсчёте. Поэтому заменим все геометрии точками. Метод `representative_point()` возвращает точку, которая гарантированно лежит внутри полигона:


In [ ]:
cafes["geometry"] = cafes.representative_point()

cafes.explore()

На карте с точками сложно оценить, где кафе больше, особенно там, где точки перекрываются. Агрегирование по сетке решает эту проблему.


## 1. Создание регулярной сетки

### 1.1. Перепроецирование

Размер ячеек задаётся в метрах, а площадь ячеек нужна в квадратных метрах, поэтому перепроецируем данные в подходящую зону UTM:


In [ ]:
utm_crs = admin_border.estimate_utm_crs()

admin_border_utm = admin_border.to_crs(utm_crs)
cafes_utm = cafes.to_crs(utm_crs)

### 1.2. Размер ячейки

Размер ячейки — ключевой параметр, который напрямую влияет на результат:

- **слишком крупные ячейки** сглаживают локальные различия, и структура распределения теряется;
- **слишком мелкие ячейки** дают много пустых ячеек и «шум».

Оптимальный размер зависит от задачи и плотности данных. Начнём с ячеек 500 × 500 м, а в разделе 3 сравним разные размеры.


### 1.3. Построение сетки

Алгоритм построения сетки простой:

1. Находим охват (_extent_) территории — минимальные и максимальные координаты по осям X и Y. В GeoPandas их возвращает атрибут `total_bounds`.
2. Начиная с левого нижнего угла, «шагаем» по X и Y с шагом, равным размеру ячейки, и на каждом шаге создаём квадрат с помощью `shapely.geometry.box()`.
3. Собираем квадраты в `GeoDataFrame` и присваиваем каждой ячейке идентификатор `cell_id`.

Оформим алгоритм в виде функции — она понадобится нам ещё не раз:


In [ ]:
from shapely.geometry import box


def create_regular_grid(data, cell_size):
    """Регулярная сетка квадратных ячеек по охвату data (в метрической СК)."""

    if data.crs is None:
        raise ValueError("У входных данных не задана система координат (CRS).")

    # Размер ячейки задаётся в метрах, поэтому нужна метрическая СК
    if data.crs.is_geographic:
        data = data.to_crs(data.estimate_utm_crs())

    minx, miny, maxx, maxy = data.total_bounds

    cells = []
    x = minx
    while x < maxx:
        y = miny
        while y < maxy:
            cells.append(box(x, y, x + cell_size, y + cell_size))
            y += cell_size
        x += cell_size

    grid = gpd.GeoDataFrame({"geometry": cells}, crs=data.crs)
    grid["cell_id"] = range(len(grid))

    return grid

In [ ]:
grid = create_regular_grid(admin_border_utm, 500)

m = grid.explore(style_kwds={"fill": False})
admin_border_utm.boundary.explore(m=m, color="red")

### 1.4. Обрезка сетки по границе территории

Сетка построена по охвату района — прямоугольнику, поэтому многие ячейки выходят за его границу. Но кафе мы загрузили только внутри района, и часть такой ячейки за границей района всегда будет «пустой». Если делить число кафе на полную площадь ячейки, плотность на краях района окажется заниженной.

Обрежем сетку по границе района. После этого краевые ячейки станут меньше, и плотность будет рассчитываться по их реальной площади:


In [ ]:
grid = grid.clip(admin_border_utm)

grid.explore(style_kwds={"fill": False})

## 2. Подсчёт точек в ячейках

### 2.1. Пространственное объединение

Определим, в какую ячейку попадает каждое кафе, — так же, как в предыдущем разделе мы определяли округ каждого театра:


In [ ]:
cafes_in_grid = gpd.sjoin(
    cafes_utm,
    grid,
    how="left",
    predicate="within"
)

print("Кафе без ячейки:", cafes_in_grid["cell_id"].isna().sum())

### 2.2. Агрегирование и присоединение к сетке

Подсчитаем количество кафе в каждой ячейке и присоединим результат к сетке. Ячейкам без кафе присвоим 0:


In [ ]:
cafes_count = (
    cafes_in_grid
    .groupby("cell_id")
    .size()
    .reset_index(name="cafe_count")
)

grid = grid.merge(cafes_count, on="cell_id", how="left")
grid["cafe_count"] = grid["cafe_count"].fillna(0).astype(int)

### 2.3. Расчёт плотности

После обрезки краевые ячейки меньше остальных, поэтому сравнивать абсолютное число кафе в ячейках уже некорректно (см. раздел [Картограммы и методы классификации](../module_3/mapping_1.ipynb)). Рассчитаем плотность — число кафе на 1 км²:


In [ ]:
grid["cell_area_km2"] = grid.geometry.area / 1_000_000
grid["cafe_density"] = grid["cafe_count"] / grid["cell_area_km2"]

При обрезке по границе могут появиться очень маленькие фрагменты ячеек. Одно кафе в фрагменте площадью 0,01 км² даёт плотность 100 кафе/км², хотя это лишь случайность. Исключим из анализа фрагменты меньше четверти полной ячейки:


In [ ]:
full_cell_km2 = 0.5 * 0.5
grid = grid[grid["cell_area_km2"] >= full_cell_km2 / 4]

grid.explore(
    column="cafe_density",
    cmap="OrRd",
    legend=True,
    legend_kwds={"caption": "Кафе на 1 км²"}
)

Кафе в Центральном районе размещены неравномерно: максимальная плотность — в западной части района, у начала Невского проспекта, и в его центральной части, а к востоку плотность заметно снижается. Такая карта гораздо нагляднее показывает распределение, чем отдельные точки.


## 3. Влияние размера ячейки

Размер ячейки — это проявление **проблемы изменяемой единицы** (MAUP), о которой мы говорили в предыдущем разделе. Сравним результаты для ячеек 250, 500 и 1000 м.

Объединим шаги из разделов 1–2 в одну функцию:


In [ ]:
def density_grid(points, border, cell_size):
    """Плотность точек (на км²) в ячейках регулярной сетки, обрезанной по границе."""
    grid = create_regular_grid(border, cell_size).clip(border)

    counts = (
        gpd.sjoin(points, grid, how="left", predicate="within")
        .groupby("cell_id")
        .size()
        .reset_index(name="count")
    )
    grid = grid.merge(counts, on="cell_id", how="left")
    grid["count"] = grid["count"].fillna(0)

    grid["area_km2"] = grid.geometry.area / 1_000_000
    grid = grid[grid["area_km2"] >= (cell_size / 1000) ** 2 / 4]
    grid["density"] = grid["count"] / grid["area_km2"]

    return grid

In [ ]:
cell_sizes = [250, 500, 1000]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, size in zip(axes, cell_sizes):
    density_grid(cafes_utm, admin_border_utm, size).plot(
        column="density",
        cmap="OrRd",
        legend=True,
        legend_kwds={"shrink": 0.6, "label": "Кафе на 1 км²"},
        ax=ax
    )
    admin_border_utm.boundary.plot(ax=ax, color="black", linewidth=0.5)
    ax.set_title(f"Ячейка {size} × {size} м")
    ax.axis("off")

plt.tight_layout()
plt.show()

Чем меньше ячейка, тем больше деталей и тем выше максимальные значения плотности: на мелких ячейках видны отдельные улицы, а на крупных — лишь общая тенденция. При этом крупные ячейки сглаживают различия между частями района.

Не существует «правильного» размера ячейки. Выбирайте его исходя из задачи и обязательно проверяйте, сохраняются ли ваши выводы при другом размере.


## Итог

В этом разделе мы:

1. привели данные к единому типу геометрии с помощью `representative_point()`;
2. построили регулярную сетку и обрезали её по границе территории;
3. подсчитали количество точек в ячейках с помощью пространственного объединения и рассчитали плотность;
4. увидели, как размер ячейки влияет на результат (MAUP).

Регулярная сетка позволяет перейти от анализа отдельных точек к выявлению пространственных закономерностей и сравнивать территории независимо от административных границ.
